# Chest X-Ray Pneumonia Classification
## Notebook 02 — Baseline CNN (from scratch)

**Goal:** Build a simple 4-block CNN trained from random initialization. This gives us a baseline to compare against the pretrained ResNet-50 in Notebook 03.

---
### Notebook roadmap
1. Rebuild DataLoaders (copy from Notebook 01)
2. Define the Baseline CNN architecture
3. Training loop with validation
4. Plot loss & accuracy curves
5. Evaluate on test set — accuracy, AUC-ROC, sensitivity, specificity
6. Confusion matrix
7. Save the best model checkpoint


## 1. Setup & DataLoaders
*(Identical to Notebook 01 — run this block first)*

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
import torchvision
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
CFG = {
    "data_root"    : Path("/content/chest_xray"),
    "img_size"     : 224,
    "batch_size"   : 32,
    "num_workers"  : 2,
    "seed"         : 42,
    "val_split"    : 0.20,
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std" : [0.229, 0.224, 0.225],
    # Training
    "epochs"       : 15,
    "lr"           : 1e-3,
    "weight_decay" : 1e-4,
}

# ── Transforms ───────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG["imagenet_mean"], std=CFG["imagenet_std"]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG["imagenet_mean"], std=CFG["imagenet_std"]),
])

# ── Dataset & splits ─────────────────────────────────────────────────────────
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

full_train_dataset = datasets.ImageFolder(
    root=str(CFG["data_root"] / "train"), transform=None
)
all_indices = list(range(len(full_train_dataset)))
all_targets = full_train_dataset.targets

train_idx, val_idx = train_test_split(
    all_indices, test_size=CFG["val_split"],
    stratify=all_targets, random_state=CFG["seed"]
)

train_ds = TransformSubset(Subset(full_train_dataset, train_idx), train_transform)
val_ds   = TransformSubset(Subset(full_train_dataset, val_idx),   val_test_transform)
test_ds  = datasets.ImageFolder(
    root=str(CFG["data_root"] / "test"), transform=val_test_transform
)

# ── WeightedRandomSampler ────────────────────────────────────────────────────
train_labels  = [all_targets[i] for i in train_idx]
class_counts  = Counter(train_labels)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          sampler=sampler, num_workers=CFG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Class mapping: {full_train_dataset.class_to_idx}")

## 2. Baseline CNN Architecture

A simple **4-block CNN**. Each block: Conv → BatchNorm → ReLU → MaxPool → Dropout.

```
Input: (B, 3, 224, 224)
  Block 1: Conv(3→32)   → BN → ReLU → Pool → (B, 32, 112, 112)
  Block 2: Conv(32→64)  → BN → ReLU → Pool → (B, 64, 56, 56)
  Block 3: Conv(64→128) → BN → ReLU → Pool → (B, 128, 28, 28)
  Block 4: Conv(128→256)→ BN → ReLU → Pool → (B, 256, 14, 14)
  AdaptiveAvgPool → (B, 256, 1, 1)
  Flatten → FC(256→128) → ReLU → Dropout → FC(128→2)
Output: (B, 2) — logits
```

In [ ]:
class ConvBlock(nn.Module):
    """Conv2d → BatchNorm2d → ReLU → MaxPool2d → Dropout2d"""
    def __init__(self, in_ch, out_ch, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=dropout),
        )
    def forward(self, x):
        return self.block(x)


class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32,  dropout=0.1),
            ConvBlock(32,  64,  dropout=0.1),
            ConvBlock(64,  128, dropout=0.2),
            ConvBlock(128, 256, dropout=0.2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # → (B, 256, 1, 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


# ── Instantiate & summarize ───────────────────────────────────────────────────
model = BaselineCNN(num_classes=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable:,}")
print()
print(model)

In [ ]:
# ── Quick forward pass sanity check ─────────────────────────────────────────
dummy = torch.randn(4, 3, 224, 224).to(DEVICE)
out   = model(dummy)
print(f"Input shape  : {dummy.shape}")
print(f"Output shape : {out.shape}   ← (batch=4, classes=2) ✓")

## 3. Training Loop

In [ ]:
# ── Loss, optimizer, scheduler ───────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

# Reduce LR by 0.5 if val loss doesn't improve for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, verbose=True
)

print("Loss      : CrossEntropyLoss")
print("Optimizer : Adam  lr={} wd={}".format(CFG["lr"], CFG["weight_decay"]))
print("Scheduler : ReduceLROnPlateau (patience=3, factor=0.5)")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

        probs = torch.softmax(logits, dim=1)[:, 1]  # P(Pneumonia)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total
    accuracy = correct / total
    auc      = roc_auc_score(all_labels, all_probs)
    return avg_loss, accuracy, auc, np.array(all_probs), np.array(all_labels)


print("Training functions defined.")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [],
           "train_acc" : [], "val_acc" : [],
           "val_auc"   : []}

best_val_loss = float("inf")
best_model_path = "/content/baseline_cnn_best.pth"

print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>9} {'Val Acc':>8} {'Val AUC':>8}")
print("-" * 60)

for epoch in range(1, CFG["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, val_auc, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        flag = " ← best"
    else:
        flag = ""

    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc:>9.3f} "
          f"{val_loss:>9.4f} {val_acc:>8.3f} {val_auc:>8.4f}{flag}")

print(f"\nBest val loss: {best_val_loss:.4f} — model saved to {best_model_path}")

## 4. Training Curves

In [ ]:
epochs = range(1, CFG["epochs"] + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(epochs, history["train_loss"], label="Train", color="#4C9EBE", linewidth=2)
axes[0].plot(epochs, history["val_loss"],   label="Val",   color="#E07B54", linewidth=2)
axes[0].set_title("Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CrossEntropy")
axes[0].legend(); axes[0].spines[["top", "right"]].set_visible(False)

# Accuracy
axes[1].plot(epochs, history["train_acc"], label="Train", color="#4C9EBE", linewidth=2)
axes[1].plot(epochs, history["val_acc"],   label="Val",   color="#E07B54", linewidth=2)
axes[1].set_title("Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1); axes[1].legend()
axes[1].spines[["top", "right"]].set_visible(False)

# AUC-ROC
axes[2].plot(epochs, history["val_auc"], color="#6BAE75", linewidth=2)
axes[2].set_title("Val AUC-ROC", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("AUC")
axes[2].set_ylim(0, 1)
axes[2].spines[["top", "right"]].set_visible(False)

fig.suptitle("Baseline CNN — Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/baseline_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_acc, test_auc, test_probs, test_labels = evaluate(
    model, test_loader, criterion, DEVICE
)

# Threshold at 0.5 for predictions
test_preds = (test_probs >= 0.5).astype(int)

# Confusion matrix components
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds).ravel()
sensitivity = tp / (tp + fn)   # recall for Pneumonia (the important one)
specificity = tn / (tn + fp)   # recall for Normal
precision   = tp / (tp + fp)
f1          = 2 * precision * sensitivity / (precision + sensitivity)

print("=" * 45)
print("  BASELINE CNN — TEST SET RESULTS")
print("=" * 45)
print(f"  Accuracy    : {test_acc:.4f}  ({test_acc*100:.1f}%)")
print(f"  AUC-ROC     : {test_auc:.4f}")
print(f"  Sensitivity : {sensitivity:.4f}  (% Pneumonia caught)")
print(f"  Specificity : {specificity:.4f}  (% Normal correctly excluded)")
print(f"  Precision   : {precision:.4f}")
print(f"  F1 Score    : {f1:.4f}")
print("=" * 45)
print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")

## 6. Confusion Matrix & ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = np.array([[tn, fp], [fn, tp]])
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Pred: Normal", "Pred: Pneumonia"], fontsize=11)
axes[0].set_yticklabels(["True: Normal", "True: Pneumonia"], fontsize=11)
axes[0].set_title("Confusion Matrix", fontweight="bold", fontsize=13)

for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                     fontsize=20, fontweight="bold", color=color)
plt.colorbar(im, ax=axes[0])

# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(test_labels, test_probs)
axes[1].plot(fpr, tpr, color="#4C9EBE", linewidth=2.5,
             label=f"Baseline CNN (AUC = {test_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Random")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
axes[1].set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
axes[1].set_title("ROC Curve", fontweight="bold", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("Baseline CNN — Test Set Evaluation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/baseline_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Save Results Summary

In [ ]:
# Save results to a dict — we'll load this in Notebook 04 for comparison
import json

baseline_results = {
    "model"       : "Baseline CNN (4-block, from scratch)",
    "accuracy"    : round(float(test_acc), 4),
    "auc_roc"     : round(float(test_auc), 4),
    "sensitivity" : round(float(sensitivity), 4),
    "specificity" : round(float(specificity), 4),
    "precision"   : round(float(precision), 4),
    "f1"          : round(float(f1), 4),
    "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
}

with open("/content/baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print("Results saved to /content/baseline_results.json")
print(json.dumps(baseline_results, indent=2))

## Summary

| Metric | Baseline CNN |
|---|---|
| Accuracy | *see output above* |
| AUC-ROC | *see output above* |
| Sensitivity | *see output above* |
| Specificity | *see output above* |

### Key things to note for writeup
- **Sensitivity > Specificity** is what we want for a medical screening tool — missing a Pneumonia case (False Negative) is more dangerous than a false alarm
- The baseline trained from scratch with ~400K parameters gives us a reference point
- **Next → Notebook 03:** Fine-tune ResNet-50 (pretrained on ImageNet, 25M parameters). We expect a significant jump in AUC-ROC and sensitivity.